# FINAL — Robust Traffic Sign Detection (`video1.mp4`)

**Một notebook duy nhất**. Mục tiêu: biển gần ổn định, biển xa được bắt sớm hơn nhưng không đổi lấy false positive.

Pipeline: **primary VTSR global + far slices + complementary YOLO11s + DIP proposals + template verification + strict current-frame temporal confirmation**.

Chọn **T4 GPU** rồi **Runtime → Run all**.


In [ ]:
# 1) Setup, Drive, persistent models
!pip install -q "ultralytics>=8.3,<9" "huggingface_hub>=0.25" "opencv-python-headless>=4.9" "requests>=2.31" "tqdm>=4.66"
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
from dataclasses import dataclass, field
from collections import defaultdict, deque
import os, shutil, subprocess, math, re, csv
import cv2, numpy as np, torch, requests
from tqdm.auto import tqdm
from huggingface_hub import hf_hub_download
from ultralytics import YOLO

ROOT=Path('/content/drive/MyDrive/DIP'); VIDEO=ROOT/'video1.mp4'
MODELS=ROOT/'models'; OUTPUTS=ROOT/'outputs'; TMPL=MODELS/'sign_templates'
P_DIR=MODELS/'traffic_sign_primary'; S_DIR=MODELS/'traffic_sign_secondary'
for p in (MODELS,OUTPUTS,TMPL,P_DIR,S_DIR): p.mkdir(parents=True,exist_ok=True)
assert torch.cuda.is_available(), 'Enable T4 GPU first'
assert VIDEO.exists(), f'Missing {VIDEO}'
print('GPU:',torch.cuda.get_device_name(0),'| video:',VIDEO)

# Remove artifacts from obsolete helmet/plate/OCR versions.
for p in [MODELS/'helmet',MODELS/'license_plate',MODELS/'scene',MODELS/'easyocr',OUTPUTS/'video1_plates',OUTPUTS/'video1_violations']:
    if p.exists(): shutil.rmtree(p,ignore_errors=True)
for p in [MODELS/'helmet_best.pt',MODELS/'plate_best.pt',OUTPUTS/'video1_result.mp4',OUTPUTS/'video1_result.csv',OUTPUTS/'video1_result_temp.mp4']:
    try: p.unlink()
    except FileNotFoundError: pass

# Persist on Drive; later Colab sessions reuse these weights (no training).
P_PATH=hf_hub_download('liamxdev/vtsr','vtsr.torchscript',local_dir=str(P_DIR))
S_PATH=hf_hub_download('star092304/traffic-sign-detection-vietnam-yolo','best.pt',local_dir=str(S_DIR))
primary=YOLO(P_PATH,task='detect'); secondary=YOLO(S_PATH); DEVICE=0
print('Primary:',P_PATH); print('Secondary:',S_PATH)


In [ ]:
# 2) Persistent template bank: old project templates + official Vietnamese sign artwork
legacy={'No Entry':['camnguocchieu.jpg','wrongway.png'],'No Parking':['noparking.png'],'No Stop/Parking':['nostopandparking.png','camdungcamdoxe.png'],'No Left Turn':['noleftturn.png','noleft.jpg'],'Keep Right':['keepright.png'],'Children':['children.png'],'Slow Down':['slow.png']}
official={'No Entry':'Vietnam road sign P102.svg','No Stop/Parking':'Vietnam road sign P130.svg','No Parking':'Vietnam road sign P131a.svg','No Left Turn':'Vietnam road sign P123a.svg','No Right Turn':'Vietnam road sign P123b.svg','No U-Turn':'Vietnam road sign P124a1.svg','Keep Right':'Vietnam road sign R302a.svg','Children':'Vietnam road sign W225.svg','Road Works':'Vietnam road sign W227.svg','No Overtaking':'Vietnam road sign P125.svg'}
def sn(s): return re.sub(r'[^A-Za-z0-9_-]+','_',s).strip('_')
def save_url(url,path):
    if path.exists() and path.stat().st_size>500:return
    try:
        r=requests.get(url,timeout=25,headers={'User-Agent':'DIP-traffic-sign-project/1.0'}); r.raise_for_status(); path.write_bytes(r.content)
    except Exception as e: print('[template warning]',e)
for label,names in legacy.items():
    for i,name in enumerate(names): save_url(f'https://raw.githubusercontent.com/NVTruong473/DIP/feature/yolo-traffic-safety/END_DIP/sign_templates/{name}',TMPL/f'{sn(label)}_legacy_{i}{Path(name).suffix}')
api='https://commons.wikimedia.org/w/api.php'
for label,name in official.items():
    out=TMPL/f'{sn(label)}_official.png'
    if out.exists() and out.stat().st_size>500:continue
    try:
        q={'action':'query','format':'json','prop':'imageinfo','iiprop':'url','iiurlwidth':320,'titles':f'File:{name}'}
        d=requests.get(api,params=q,timeout=25,headers={'User-Agent':'DIP-traffic-sign-project/1.0'}).json(); page=next(iter(d['query']['pages'].values()))
        save_url(page['imageinfo'][0].get('thumburl') or page['imageinfo'][0]['url'],out)
    except Exception as e: print('[Commons warning]',label,e)
print('Template bank:',len(list(TMPL.glob('*'))),'files ->',TMPL)


In [ ]:
# 3) Detection + validation + current-frame-only stabilization
COMMON={'P-102':'No Entry','P-123A':'No Left Turn','P-123B':'No Right Turn','P-124A':'No U-Turn','P-127':'Speed Limit','P-130':'No Stop/Parking','P-131A':'No Parking','P-245A':'Slow Down','R-302A':'Keep Right','R-302B':'Keep Left','R-303':'Roundabout','R-407A':'One Way','W-224':'Pedestrian Crossing','W-225':'Children','W-227':'Road Works'}
SHORT={'No Stopping & No Parking':'No Stop/Parking','Children Crossing':'Children','Road Work Ahead':'Road Works','No U-Turn and No Left Turn':'No U-Turn/Left','No U-Turn and No Right Turn':'No U-Turn/Right','Intersection with a Minor Road':'Minor Junction','Intersection with Equal Roads':'Equal Junction','Intersection with a Priority Road':'Priority Junction','Level Crossing with Barriers':'Rail Crossing','No Two or Three-wheeled Vehicles':'No 2/3-Wheelers'}
EXCLUDE={'Green Light','Red Light'}
@dataclass
class Det:
    box:tuple; label:str; conf:float; source:str; family:str; color:float=0.; tmpl:float=0.; score:float=0.
def canon(s):
    s=str(s).upper().replace('_','-').replace('.','-')
    while '--' in s:s=s.replace('--','-')
    return s
def plabel(raw):
    c=canon(raw)
    if c in COMMON:return COMMON[c]
    if c.startswith('P-'):return f'Prohibition {c}'
    if c.startswith('R-'):return f'Mandatory {c}'
    if c.startswith('W-'):return f'Warning {c}'
    return f'Traffic Sign {c}'
def slabel(raw):
    s=re.sub(r'\s+',' ',str(raw)).strip(); return SHORT.get(s,s[:28])
def iou(a,b):
    x1=max(a[0],b[0]);y1=max(a[1],b[1]);x2=min(a[2],b[2]);y2=min(a[3],b[3]); inter=max(0,x2-x1)*max(0,y2-y1)
    if inter<=0:return 0.
    aa=max(1,a[2]-a[0])*max(1,a[3]-a[1]);bb=max(1,b[2]-b[0])*max(1,b[3]-b[1]);return inter/(aa+bb-inter)
def plausible(b,W,H):
    w=max(1,b[2]-b[0]);h=max(1,b[3]-b[1]);return min(w,h)>=6 and .18<=w/h<=5.5 and w*h/(W*H)<.14
def color_support(crop):
    if crop.size==0:return 0.
    h=cv2.cvtColor(crop,cv2.COLOR_BGR2HSV); ms=[cv2.inRange(h,(0,60,45),(12,255,255)),cv2.inRange(h,(165,60,45),(180,255,255)),cv2.inRange(h,(88,55,40),(138,255,255)),cv2.inRange(h,(14,55,55),(42,255,255))]
    m=ms[0]
    for z in ms[1:]:m=cv2.bitwise_or(m,z)
    return np.count_nonzero(m)/m.size
def enhance(img):
    lab=cv2.cvtColor(img,cv2.COLOR_BGR2LAB);l,a,b=cv2.split(lab);l=cv2.createCLAHE(1.8,(8,8)).apply(l);x=cv2.cvtColor(cv2.merge([l,a,b]),cv2.COLOR_LAB2BGR);blur=cv2.GaussianBlur(x,(0,0),1);return cv2.addWeighted(x,1.15,blur,-.15,0)

# Edge templates. They can only VERIFY the class already proposed by YOLO.
TF=defaultdict(list)
for p in TMPL.glob('*'):
    img=cv2.imread(str(p)); label=None
    if img is None:continue
    for k in official:
        if p.name.startswith(sn(k)):label=k;break
    if label is None:
        for k in legacy:
            if p.name.startswith(sn(k)):label=k;break
    if not label:continue
    g=cv2.cvtColor(cv2.resize(img,(72,72)),cv2.COLOR_BGR2GRAY)
    for ang in (-6,0,6):
        M=cv2.getRotationMatrix2D((36,36),ang,1);r=cv2.warpAffine(g,M,(72,72),borderMode=cv2.BORDER_REPLICATE);e=cv2.Canny(cv2.createCLAHE(1.5,(8,8)).apply(r),55,150).astype(np.float32);e=(e-e.mean())/(e.std()+1e-6);TF[label].append(e)
def template_score(crop,label):
    if label not in TF or crop.size==0:return 0.
    g=cv2.cvtColor(cv2.resize(crop,(72,72)),cv2.COLOR_BGR2GRAY);e=cv2.Canny(cv2.createCLAHE(1.5,(8,8)).apply(g),55,150).astype(np.float32);e=(e-e.mean())/(e.std()+1e-6);return max(float((e*t).mean()) for t in TF[label])
def parse(result,off,frame,src,fam):
    ox,oy=off;H,W=frame.shape[:2];out=[];names=result.names
    if result.boxes is None:return out
    for b in result.boxes:
        x1,y1,x2,y2=map(int,b.xyxy[0].tolist());bb=(max(0,x1+ox),max(0,y1+oy),min(W-1,x2+ox),min(H-1,y2+oy))
        if not plausible(bb,W,H):continue
        cls=int(b.cls[0]);raw=str(names.get(cls,cls) if isinstance(names,dict) else names[cls])
        if fam=='s' and raw in EXCLUDE:continue
        label=plabel(raw) if fam=='p' else slabel(raw);conf=float(b.conf[0]);crop=frame[bb[1]:bb[3],bb[0]:bb[2]];cs=color_support(crop);ts=template_score(crop,label) if conf<.40 else 0.
        score=conf+(.055 if fam=='p' else 0)+(.025 if 'global' in src else 0)+.10*max(0,ts)+.035*min(1,cs*6)
        out.append(Det(bb,label,conf,src,fam,cs,ts,score))
    return out
def infer(model,img,frame,src,fam,conf,size,off=(0,0)):
    return parse(model.predict(img,conf=conf,imgsz=size,device=DEVICE,verbose=False,max_det=70)[0],off,frame,src,fam)
def far_tiles(frame):
    H,W=frame.shape[:2];y2=int(H*.78);tw=int(W*.62);return [(frame[:y2,:tw],(0,0),'far_left'),(frame[:y2,W-tw:],(W-tw,0),'far_right')]
def dip_props(frame):
    H,W=frame.shape[:2];y2=int(H*.80);roi=frame[:y2];hsv=cv2.cvtColor(roi,cv2.COLOR_BGR2HSV);ms=[cv2.inRange(hsv,(0,75,45),(12,255,255)),cv2.inRange(hsv,(165,75,45),(180,255,255)),cv2.inRange(hsv,(90,65,45),(138,255,255)),cv2.inRange(hsv,(15,70,55),(40,255,255))];m=ms[0]
    for z in ms[1:]:m=cv2.bitwise_or(m,z)
    m=cv2.morphologyEx(m,cv2.MORPH_OPEN,np.ones((3,3),np.uint8));cnts,_=cv2.findContours(m,cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE);cand=[]
    for c in cnts:
        a=cv2.contourArea(c)
        if not 18<a<9000:continue
        x,y,w,h=cv2.boundingRect(c);per=cv2.arcLength(c,True);circ=4*math.pi*a/(per*per+1e-6)
        if min(w,h)<5 or max(w,h)>150 or not .45<w/max(1,h)<1.75:continue
        if circ<.35 and not 3<=len(cv2.approxPolyDP(c,.04*per,True))<=8:continue
        cx=x+w/2;cy=y+h/2;side=int(max(54,max(w,h)*3.2));bb=(max(0,int(cx-side/2)),max(0,int(cy-side/2)),min(W,int(cx+side/2)),min(y2,int(cy+side/2)));ratio=np.count_nonzero(m[y:y+h,x:x+w])/max(1,w*h);cand.append((ratio+a/9000,bb))
    cand.sort(reverse=True);out=[]
    for _,bb in cand:
        if any(iou(bb,k)>.35 for k in out):continue
        out.append(bb)
        if len(out)==2:break
    return out
def merge(ds):
    out=[]
    for d in sorted(ds,key=lambda x:x.score,reverse=True):
        matched=False
        for i,k in enumerate(out):
            if iou(d.box,k.box)<.48:continue
            if d.label==k.label:
                if d.score>k.score:out[i]=d
            elif d.score+(.035 if d.family=='p' else 0)>k.score+(.035 if k.family=='p' else 0)+.025:out[i]=d
            matched=True;break
        if not matched:out.append(d)
    return out
@dataclass
class Track:
    label:str;box:tuple;last:int;hits:deque=field(default_factory=lambda:deque(maxlen=7))
class Stabilizer:
    def __init__(self):self.t={};self.n=1
    def update(self,ds,f):
        for i in list(self.t):
            if f-self.t[i].last>6:del self.t[i]
        shown=[];used=set()
        for d in sorted(ds,key=lambda x:x.score,reverse=True):
            best=None;bm=0.
            for i,t in self.t.items():
                if i in used or t.label!=d.label:continue
                ov=iou(d.box,t.box);dc=((d.box[0]+d.box[2])/2,(d.box[1]+d.box[3])/2);tc=((t.box[0]+t.box[2])/2,(t.box[1]+t.box[3])/2);diag=max(8.,math.hypot(d.box[2]-d.box[0],d.box[3]-d.box[1]));dist=math.hypot(dc[0]-tc[0],dc[1]-tc[1])/diag;match=max(ov,.22 if dist<.55 else 0.)
                if match>bm:bm=match;best=i
            if best is None or bm<.18:
                best=self.n;self.n+=1;self.t[best]=Track(d.label,d.box,f)
            t=self.t[best];used.add(best);a=.46 if max(d.box[2]-d.box[0],d.box[3]-d.box[1])<45 else .64;t.box=tuple(int(a*n+(1-a)*o) for n,o in zip(d.box,t.box));t.last=f;t.hits.append((f,d.score,d.conf,d.tmpl,d.color));recent=[x for x in t.hits if f-x[0]<=4]
            strong=d.score>=.62 or d.conf>=.58;normal=d.conf>=.24 and len(recent)>=2;weak=d.conf>=.14 and len(recent)>=3 and (d.tmpl>=.42 or d.color>=.085)
            if strong or normal or weak:d.box=t.box;shown.append(d)
        return shown
def draw(frame,b,text):
    x1,y1,x2,y2=map(int,b);col=(0,165,255);cv2.rectangle(frame,(x1,y1),(x2,y2),col,2);font=cv2.FONT_HERSHEY_SIMPLEX;sc=.52;th=2;(tw,hh),base=cv2.getTextSize(text,font,sc,th);pad=5;lh=hh+base+2*pad
    if y1>=lh:top=y1-lh;bottom=y1;ty=bottom-base-pad
    else:top=y1;bottom=min(frame.shape[0]-1,y1+lh);ty=top+hh+pad
    cv2.rectangle(frame,(x1,top),(min(frame.shape[1]-1,x1+tw+2*pad),bottom),col,-1);cv2.putText(frame,text,(x1+pad,max(hh,ty)),font,sc,(255,255,255),th,cv2.LINE_AA)


In [ ]:
# 4) Run and save H.264 result to Drive
cap=cv2.VideoCapture(str(VIDEO));assert cap.isOpened();fps=float(cap.get(cv2.CAP_PROP_FPS)) or 30.;total=int(cap.get(cv2.CAP_PROP_FRAME_COUNT));W=int(cap.get(cv2.CAP_PROP_FRAME_WIDTH));H=int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
OUT=OUTPUTS/'video1_result.mp4';TEMP=OUTPUTS/'video1_result_temp.mp4';CSV=OUTPUTS/'video1_result.csv';wr=cv2.VideoWriter(str(TEMP),cv2.VideoWriter_fourcc(*'mp4v'),fps,(W,H));assert wr.isOpened()
stab=Stabilizer();rows=[];f=0;bar=tqdm(total=total,desc='Traffic signs: global + far rescue',unit='frame')
while True:
    ok,frame=cap.read()
    if not ok:break
    ds=infer(primary,frame,frame,'p_global','p',.17,768)
    for tile,off,name in far_tiles(frame):ds+=infer(primary,enhance(tile),frame,'p_'+name,'p',.13,896,off)
    if f%3==0:
        ds+=infer(secondary,frame,frame,'s_global','s',.20,704);top=enhance(frame[:int(H*.76)]);ds+=infer(secondary,top,frame,'s_far','s',.16,768)
        for j,bb in enumerate(dip_props(frame)):
            x1,y1,x2,y2=bb;ds+=infer(primary,enhance(frame[y1:y2,x1:x2]),frame,f'dip_{j}','p',.10,512,(x1,y1))
    visible=stab.update(merge(ds),f)
    for d in visible:
        draw(frame,d.box,d.label);rows.append([f,round(f/fps,3),d.label,round(d.conf,4),*d.box,d.source,round(d.tmpl,3),round(d.color,3),round(d.score,3)])
    wr.write(frame);f+=1;bar.update(1)
bar.close();cap.release();wr.release()
with open(CSV,'w',newline='',encoding='utf-8') as fp:
    w=csv.writer(fp);w.writerow(['frame','time_sec','class','confidence','x1','y1','x2','y2','source','template_score','color_score','effective_score']);w.writerows(rows)
try:
    subprocess.run(['ffmpeg','-y','-loglevel','error','-i',str(TEMP),'-i',str(VIDEO),'-map','0:v:0','-map','1:a?','-c:v','libx264','-preset','veryfast','-crf','22','-pix_fmt','yuv420p','-tag:v','avc1','-movflags','+faststart','-c:a','aac','-shortest',str(OUT)],check=True)
except Exception:
    subprocess.run(['ffmpeg','-y','-loglevel','error','-i',str(TEMP),'-c:v','libx264','-crf','22','-pix_fmt','yuv420p','-an',str(OUT)],check=True)
try:TEMP.unlink()
except:pass
print('Saved:',OUT);print('CSV:',CSV,'rows=',len(rows))


In [ ]:
# 5) Preview directly in Colab
from IPython.display import Video,display
PREVIEW=Path('/content/video1_result_preview.mp4')
subprocess.run(['ffmpeg','-y','-loglevel','error','-i',str(OUT),'-vf','scale=960:-2','-c:v','libx264','-preset','veryfast','-crf','29','-pix_fmt','yuv420p','-an',str(PREVIEW)],check=True)
display(Video(str(PREVIEW),embed=True,width=960,html_attributes='controls'))


## Persistence
Weights và template bank nằm trong `MyDrive/DIP/models/`. Nếu Colab disconnect, mở lại notebook và **Run all**; model không train lại, chỉ inference video chạy lại.
